# 03 — Student distillation (6-layer XLM-R)

Distills the fine-tuned `teacher-v1` checkpoint (from `training/02_teacher_finetune.ipynb`) into a 6-layer student: initialized by copying the teacher's embeddings, every other transformer layer, and the classification head, then trained on temperature-softened KD against the teacher's logits (T=2.0) blended 0.7/0.3 with the real hard labels.

**Runtime:** Colab Pro, GPU runtime set to A100 or H100.

## Setup

In [ ]:
# git clone
!git clone --depth 1 https://github.com/gjvarun0307/real-time-moderation-pipeline.git /content/repo

import sys

sys.path.insert(0, "/content/repo/src")

In [ ]:
# install requirements
!pip install -q "transformers>=4.51" "sentencepiece>=0.2" "scikit-learn>=1.5" \
    "pyarrow>=17.0" "homoglyphs>=2.0" "regex>=2024.5.15"

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/moderation-pipeline/data/processed")
CHECKPOINT_ROOT = Path("/content/drive/MyDrive/moderation-pipeline/checkpoints")
TEACHER_CHECKPOINT_DIR = CHECKPOINT_ROOT / "teacher-v1"
STUDENT_CHECKPOINT_DIR = CHECKPOINT_ROOT / "student-v1"
STUDENT_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## Config

In [ ]:
import torch

MAX_SEQ_LEN = 192  # same value the teacher used
BATCH_SIZE = 128
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
POS_WEIGHT_CAP = 50.0  # caps class-weighted BCE for the rarest label (threat)
GRAD_CLIP_NORM = 1.0
TEMPERATURE = 2.0  # KD softmax/sigmoid temperature
ALPHA_KD = 0.7  # weight on the soft (teacher) loss; hard-label loss gets 1 - ALPHA_KD
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "no GPU detected — Runtime > Change runtime type > GPU (A100/H100)"
print(f"device: {DEVICE}, {torch.cuda.get_device_name(0)}")

torch.manual_seed(SEED)

## Load prepared data

In [ ]:
import pandas as pd

train_df = pd.read_parquet(DATA_DIR / "train_en.parquet")
val_df = pd.read_parquet(DATA_DIR / "val_en.parquet")
eval_ml_df = pd.read_parquet(DATA_DIR / "eval_multilingual.parquet")

print(f"train_en: {len(train_df)}  val_en: {len(val_df)}  eval_multilingual: {len(eval_ml_df)}")

## Load teacher, build student



In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from training.student_model import LAYERS_TO_KEEP, build_student_from_teacher

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_CHECKPOINT_DIR)
teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER_CHECKPOINT_DIR).to(
    DEVICE
)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

student = build_student_from_teacher(teacher, layers_to_keep=LAYERS_TO_KEEP).to(DEVICE)

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())
print(f"teacher: {teacher.config.num_hidden_layers} layers, {teacher_params:,} params")
print(f"student: {student.config.num_hidden_layers} layers, {student_params:,} params")

## Datasets, dataloaders, class weights

In [ ]:
from torch.utils.data import DataLoader

from training.data_prep import LABEL_COLUMNS
from training.teacher_model import ToxicityDataset, compute_pos_weight, make_collate_fn

collate_fn = make_collate_fn(teacher_tokenizer, MAX_SEQ_LEN)

train_ds = ToxicityDataset(train_df)
val_ds = ToxicityDataset(val_df)
# eval_multilingual only carries the single "toxic" label
eval_ml_ds = ToxicityDataset(eval_ml_df, label_columns=["toxic"])

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
eval_ml_loader = DataLoader(
    eval_ml_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

pos_weight = compute_pos_weight(train_df, cap=POS_WEIGHT_CAP).to(DEVICE)
pos_weight_by_label = dict(zip(LABEL_COLUMNS, pos_weight.tolist(), strict=True))
print("pos_weight per label:", pos_weight_by_label)

## Optimizer, schedule

In [ ]:
from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
num_training_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * num_training_steps),
    num_training_steps=num_training_steps,
)

## Training loop

Same raw-loop shape as `02_teacher_finetune`, extended with the KD term: each step runs a no-grad forward pass through the frozen teacher to get its logits, then `distillation_loss` blends the temperature-softened KD loss against those logits with the class-weighted hard-label loss.

In [ ]:
from training.student_model import distillation_loss
from training.teacher_model import evaluate

step = 0
for epoch in range(EPOCHS):
    student.train()
    running_loss = 0.0
    for batch in train_loader:
        labels = batch.pop("labels").to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            teacher_logits = teacher(**batch).logits

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            student_logits = student(**batch).logits
            loss = distillation_loss(
                student_logits,
                teacher_logits,
                labels,
                pos_weight,
                temperature=TEMPERATURE,
                alpha=ALPHA_KD,
            )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        step += 1
        if step % 200 == 0:
            avg_loss = running_loss / 200
            print(f"epoch {epoch + 1} step {step}/{num_training_steps} loss={avg_loss:.4f}")
            running_loss = 0.0

    val_scores = evaluate(student, val_loader, DEVICE)
    print(f"\n=== epoch {epoch + 1} val_en PR-AUC (student) ===")
    for label, score in val_scores.items():
        print(f"  {label}: {score:.4f}")
    print()

## Final evaluation

In [ ]:
final_val_scores = evaluate(student, val_loader, DEVICE)
final_ml_scores = evaluate(student, eval_ml_loader, DEVICE, label_columns=["toxic"])

per_lang_scores = {}
for lang in sorted(eval_ml_df["lang"].unique()):
    lang_df = eval_ml_df[eval_ml_df["lang"] == lang]
    lang_ds = ToxicityDataset(lang_df, label_columns=["toxic"])
    lang_loader = DataLoader(
        lang_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
    )
    per_lang_scores[lang] = evaluate(student, lang_loader, DEVICE, label_columns=["toxic"])

print("val_en (English, held-out):", final_val_scores)
print("eval_multilingual (es+it+tr combined, toxic-only):", final_ml_scores)
print("per-language toxic PR-AUC:", per_lang_scores)

In [ ]:
import json

metrics = {
    "model_name": "xlm-roberta-base",
    "teacher_checkpoint": str(TEACHER_CHECKPOINT_DIR),
    "layers_to_keep": list(LAYERS_TO_KEEP),
    "max_seq_len": MAX_SEQ_LEN,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "temperature": TEMPERATURE,
    "alpha_kd": ALPHA_KD,
    "pos_weight_cap": POS_WEIGHT_CAP,
    "teacher_param_count": teacher_params,
    "student_param_count": student_params,
    "val_en_pr_auc": final_val_scores,
    "eval_multilingual_toxic_pr_auc": final_ml_scores,
    "eval_multilingual_toxic_pr_auc_by_lang": per_lang_scores,
}
with open(STUDENT_CHECKPOINT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nwrote {STUDENT_CHECKPOINT_DIR / 'metrics.json'}")

## Save checkpoint

In [ ]:
student.save_pretrained(STUDENT_CHECKPOINT_DIR)
teacher_tokenizer.save_pretrained(STUDENT_CHECKPOINT_DIR)
print(f"saved student checkpoint + tokenizer to {STUDENT_CHECKPOINT_DIR}")

## Next

This checkpoint and `metrics.json` are both in `STUDENT_CHECKPOINT_DIR` on Drive. 

}